# ДЗ 2. Поиск утечек данных

Предсказываем опоздание заказа в момент его оформления.
Таргет: late_delivery, где 1 — опоздал, 0 — доставлен вовремя.

Нужно найти три утечки, объяснить их происхождение и сравнить
ROC-AUC до и после удаления.

Во всех экспериментах:
- HistGradientBoostingClassifier(random_state=42), остальные параметры по умолчанию;
- одно случайное разбиение 75/25 со стратификацией;
- ROC-AUC по вероятностям класса 1;
- без подбора параметров и создания новых признаков.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

SEED = 42

df = pd.read_csv(
    "data/olist_delivery_export.csv",
    parse_dates=["ordered_at"]
)

category_columns = [
    "client_state",
    "partner_state",
    "category",
    "pay_method"
]

for column in category_columns:
    df[column] = df[column].astype("category")

print("Размер:", df.shape)
display(df.head())
display(df.isna().sum())
display(df["late_delivery"].value_counts())

Размер: (96470, 29)


,ordered_at,client_key,partner_id,client_state,partner_state,intra_state,dist_km,promised_days,dispatch_deadline_days,items_qty,...,installments,paid_value,late_delivery,order_hour,order_weekday,photos_qty,title_len,handover_lag_days,seller_score,record_id
0,2018-03-03 17:39:37,568843f1b07989b16ad67462e4f7eddf,0bae85eb84b9fb3bd773911e89288d54,SC,SP,0,82.424278,11.264155,5.007500,1,...,1.0,49.17,0,17,5,4.0,47.0,1.977211,0.9923,125097
1,2018-07-15 22:36:39,7fe36cf7ee4c8a68c57dbf47d31c952c,87d3c3aeb3ead335511b3ce315eb341e,PI,SP,0,2093.948450,16.057882,4.009479,1,...,1.0,127.46,0,22,6,1.0,28.0,0.604410,0.9333,108873
2,2017-11-25 00:55:18,3e7688aca4163064b392dba8b3184807,e5a3438891c0bfdb9394643f95273d8e,SP,SP,1,166.125935,15.961597,9.471771,1,...,1.0,35.41,0,0,5,NaN,NaN,9.846748,0.8122,124268
3,2017-05-21 23:24:21,9bab4772c790be2269300fe0d3a0ed69,5cf13accae3222c70a9cac40818ae839,RJ,MG,0,568.116589,22.024757,7.170949,1,...,1.0,95.01,0,23,6,2.0,51.0,1.482477,0.9660,139775
4,2018-08-06 09:09:46,839a04051a2c347397fd425d01fc3eb2,1fe5540d7c1c37a595fefbacd5570d9e,PR,RJ,0,1204.797590,35.618218,8.010266,2,...,2.0,378.38,0,9,0,1.0,51.0,1.205718,0.9500,130113


ordered_at                   0
client_key                   0
partner_id                   0
client_state                 0
partner_state                0
intra_state                  0
dist_km                    476
promised_days                0
dispatch_deadline_days       0
items_qty                    0
partners_qty                 0
skus_qty                     0
goods_value                  0
freight_value                0
freight_ratio                0
category                  1358
weight_g                    16
volume_cm3                  16
pay_method                   1
installments                 1
paid_value                   1
late_delivery                0
order_hour                   0
order_weekday                0
photos_qty                1358
title_len                 1358
handover_lag_days            1
seller_score                 0
record_id                    0
dtype: int64

late_delivery
0    88644
1     7826
Name: count, dtype: int64

In [2]:
excluded_columns = [
    "ordered_at",
    "client_key",
    "partner_id",
    "late_delivery"
]

X = df.drop(columns=excluded_columns)
y = df["late_delivery"]

all_features = X.columns.tolist()

train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.25,
    stratify=y,
    random_state=SEED
)

display(pd.DataFrame({
    "rows": [len(train_idx), len(test_idx)],
    "late_share": [
        y.loc[train_idx].mean(),
        y.loc[test_idx].mean()
    ]
}, index=["train", "test"]))

print("Признаков:", len(all_features))

,rows,late_share
train,72352,0.081117
test,24118,0.081143


Признаков: 25


In [3]:
def evaluate_model(columns):
    model = HistGradientBoostingClassifier(random_state=SEED)

    model.fit(
        X.loc[train_idx, columns],
        y.loc[train_idx]
    )

    probabilities = model.predict_proba(
        X.loc[test_idx, columns]
    )[:, 1]

    return roc_auc_score(y.loc[test_idx], probabilities)


baseline_auc = evaluate_model(all_features)
print("Исходный ROC-AUC:", round(baseline_auc, 6))

Исходный ROC-AUC: 0.911209


## Проверка отдельных признаков

Для каждого числового признака вычисляю ROC-AUC, используя его значения
как оценки риска опоздания - без обучения отдельной модели.

Строки с пропусками исключаем только из этой проверки.
Исходные данные и разбиение train/test не меняем.


In [6]:
feature_results = []

for column in df.select_dtypes(include="number").columns:
    if column == "late_delivery":
        continue

    available = df[column].notna()

    auc = roc_auc_score(
        y.loc[available],
        df.loc[available, column]
    )

    feature_results.append({
        "column": column,
        "n_valid": int(available.sum()),
        "missing": int((~available).sum()),
        "roc_auc": auc,
        "auc_any_direction": max(auc, 1 - auc)
    })

feature_auc = pd.DataFrame(feature_results)

display(
    feature_auc
    .sort_values("auc_any_direction", ascending=False)
    .round(6)
)

,column,n_valid,missing,roc_auc,auc_any_direction
20,record_id,96470,0,0.799514,0.799514
19,seller_score,96470,0,0.298686,0.701314
18,handover_lag_days,96469,1,0.648531,0.648531
1,dist_km,95994,476,0.562776,0.562776
2,promised_days,96470,0,0.450509,0.549491
0,intra_state,96470,0,0.450552,0.549448
8,freight_value,96470,0,0.546375,0.546375
13,paid_value,96469,1,0.526051,0.526051
7,goods_value,96470,0,0.521316,0.521316
10,weight_g,96454,16,0.518712,0.518712


In [7]:
display(
    df.groupby("late_delivery")["handover_lag_days"]
    .agg(["count", "min", "median", "mean", "max"])
)

print("Пропусков:", df["handover_lag_days"].isna().sum())
print("Отрицательных значений:", df["handover_lag_days"].lt(0).sum())

,count,min,median,mean,max
late_delivery,,,,,
0,88644,-171.212419,2.138270,2.997441,55.101030
1,7825,-0.035197,3.437419,5.833761,125.775521


Пропусков: 1
Отрицательных значений: 165


## Утечка 1: handover_lag_days

По смыслу это фактический интервал до передачи перевозчику.
В момент оформления будущая передача ещё не произошла, поэтому её
фактическое время неизвестно.

Признак сообщает модели информацию о дальнейшем ходе доставки.
Удаляем его. Вместо него можно было бы использовать заранее известный
плановый срок или статистику ранее завершённых заказов.

In [8]:
id_group = df["record_id"].ge(500000).map({
    False: "Нижний диапазон",
    True: "Верхний диапазон"
})

display(
    df.groupby(id_group)["record_id"]
    .agg(["min", "max", "count"])
)

display(pd.crosstab(id_group, y))

display(
    pd.crosstab(id_group, y, normalize="index").round(6)
)

id_inside_range = df["record_id"] % 100000

print(
    "AUC исходного ID:",
    round(roc_auc_score(y, df["record_id"]), 6)
)

print(
    "AUC номера без префикса:",
    round(roc_auc_score(y, id_inside_range), 6)
)

,min,max,count
record_id,,,
Верхний диапазон,900000,949997,23910
Нижний диапазон,100000,149996,72560


late_delivery,0,1
record_id,,
Верхний диапазон,17649,6261
Нижний диапазон,70995,1565


late_delivery,0,1
record_id,,
Верхний диапазон,0.738143,0.261857
Нижний диапазон,0.978432,0.021568


AUC исходного ID: 0.799514
AUC номера без префикса: 0.501958


## Утечка 2: record_id

У технического номера два диапазона. В верхнем доля опозданий около
26.19%, в нижнем — около 2.16%.

AUC исходного номера около 0.800, а после удаления префикса —
около 0.502. Основную информацию несёт выбор диапазона.

исход заказа повлиял на выбор
партии или диапазона номеров.

Удаляем record_id. Для связи записей можно использовать независимый от
исхода ID, назначенный при создании заказа. Использовать техническую
нумерацию как числовой признак здесь не нужно.

In [9]:
reconstructed_score = (
    1 - df.groupby("partner_id")["late_delivery"].transform("mean")
).round(4)

matches = df["seller_score"].eq(reconstructed_score)

print("Доля совпадений:", matches.mean())
print("Количество несовпадений:", (~matches).sum())

orders_per_seller = (
    df.groupby("partner_id")["late_delivery"].transform("size")
)

single_order_sellers = df.loc[orders_per_seller == 1]

print("Продавцов с одним заказом:", len(single_order_sellers))

display(pd.crosstab(
    single_order_sellers["late_delivery"],
    single_order_sellers["seller_score"]
))

Доля совпадений: 1.0
Количество несовпадений: 0
Продавцов с одним заказом: 541


seller_score,0.0,1.0
late_delivery,,
0,0,488
1,53,0


## Утечка 3: seller_score

seller_score точно совпадает с долей своевременных доставок продавца,
рассчитанной по всей выгрузке и округлённой до четырёх знаков.

Агрегат включает исход текущего заказа и исходы заказов из test.
У продавца с единственным заказом score напрямую раскрывает его исход:
1 для своевременного заказа и 0 для опоздавшего.

Удаляем признак. 



In [10]:
leak_columns = [
    "handover_lag_days",
    "record_id",
    "seller_score"
]

results = [{
    "variant": "Исходная выгрузка",
    "n_features": len(all_features),
    "roc_auc": baseline_auc
}]

for leak in leak_columns:
    columns = [
        column
        for column in all_features
        if column != leak
    ]

    auc = evaluate_model(columns)

    results.append({
        "variant": f"Без {leak}",
        "n_features": len(columns),
        "roc_auc": auc
    })

display(pd.DataFrame(results).round(6))

,variant,n_features,roc_auc
0,Исходная выгрузка,25,0.911209
1,Без handover_lag_days,24,0.891885
2,Без record_id,24,0.821679
3,Без seller_score,24,0.897571
